# **Inital EDA**
### **Orders Dataset - Initial Data Profiling**

In [139]:
import pandas as pd
from pathlib import Path
import numpy as np

In [140]:
#Define the project root and raw data directory
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

In [141]:
#Read the orders CSV file into a DataFrame
df_orders = pd.read_csv(RAW_DATA_DIR / "olist_orders_dataset.csv", dtype=str)

In [142]:
df_orders.shape

(99441, 8)

In [143]:
df_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [144]:
df_orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [145]:
df_orders['order_id'].is_unique
df_orders['order_id'].isnull().sum()

np.int64(0)

**Dataset Overview**

The orders dataset contains 99.441 rows and 8 columns. It stores order identifiers, order status, and timestamps associated with the order lifecycle.

**Grain and Key Validation**

The dataset grain is one row per order.

The `order_id` column is a validated primary key candidate for the current dataset:

- *Uniqueness:* all 99,441 values are distinct.
- *Completeness:* the column contains no missing values.

These checks support the expected grain of one row per order.

**Missing Values**

Missing values were identified in the following timestamp columns:

- `order_approved_at`
- `order_delivered_carrier_date`
- `order_delivered_customer_date`

These missing values require further investigation against `order_status` to distinguish expected lifecycle states from potential data quality issues.

No missing values have been filled, and no rows have been removed at this stage.



In [146]:
na_order_approved_at = df_orders['order_approved_at'].isna().groupby(df_orders['order_status']).sum()
na_order_delivered_carrier_date = df_orders['order_delivered_carrier_date'].isna().groupby(df_orders['order_status']).sum()
na_order_delivered_customer_date = df_orders['order_delivered_customer_date'].isna().groupby(df_orders['order_status']).sum()
orders_by_status = df_orders['order_status'].value_counts()

In [147]:
#Investigating the number of missing values by order status for the columns 'order_approved_at', 'order_delivered_carrier_date', and 'order_delivered_customer_date'
na_comparision_by_status = pd.concat([orders_by_status.rename("total_orders"),
                                     na_order_approved_at.rename("na_order_approved_at"),
                                      na_order_delivered_carrier_date.rename("na_order_delivered_carrier_date"),
                                      na_order_delivered_customer_date.rename("na_order_delivered_customer_date")
                                    ],  
                                      axis=1)

na_comparision_by_status

,total_orders,na_order_approved_at,na_order_delivered_carrier_date,na_order_delivered_customer_date
order_status,,,,
delivered,96478,14,2,8
shipped,1107,0,0,1107
canceled,625,141,550,619
unavailable,609,0,609,609
invoiced,314,0,314,314
processing,301,0,301,301
created,5,5,5,5
approved,2,0,2,2


In [148]:
#Investgating the delivered orders with missing values in the 'order_delivered_customer_date' column
delivered_without_delivery_date = df_orders[(df_orders['order_delivered_customer_date'].isna()) & (df_orders['order_status'] == 'delivered')]
delivered_without_delivery_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


### Delivered Orders with Missing Delivery Timestamps

Eight orders have a `delivered` status but no `order_delivered_customer_date`. Of these, one also lacks `order_delivered_carrier_date`. The approval and estimated delivery timestamps are populated for all eight orders.

In [149]:
canceled_orders_with_delivery_date = df_orders[(df_orders['order_delivered_customer_date'].notna()) & (df_orders['order_status'] == 'canceled')]
canceled_orders_with_delivery_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29 00:00:00
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30 00:00:00
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25 00:00:00
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08 00:00:00


In [150]:
canceled_orders_analysis =canceled_orders_with_delivery_date.copy()

timestamp_columns = ['order_purchase_timestamp',
                                    'order_approved_at',
                                    'order_delivered_carrier_date',
                                    'order_delivered_customer_date',
                                    'order_estimated_delivery_date']

for header in timestamp_columns:
    canceled_orders_analysis[header] = pd.to_datetime(canceled_orders_analysis[header], errors='raise')

canceled_orders_analysis.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [151]:
purchase_before_approval = (
    canceled_orders_analysis["order_purchase_timestamp"]
    <= canceled_orders_analysis["order_approved_at"]
)

approval_before_delivered_carrier = (
    canceled_orders_analysis["order_approved_at"]
    <= canceled_orders_analysis["order_delivered_carrier_date"]
)

delivered_carrier_before_delivered_customer = (
    canceled_orders_analysis["order_delivered_carrier_date"]
    <= canceled_orders_analysis["order_delivered_customer_date"]
)

valid_lifecycle_sequence = (
    purchase_before_approval
    & approval_before_delivered_carrier
    & delivered_carrier_before_delivered_customer
).sum()

valid_lifecycle_sequence

np.int64(6)

### Canceled Orders with Recorded Delivery

Six orders have a `canceled` status despite having all lifecycle timestamps populated.

All six records satisfy the expected chronological sequence: purchase → approval → carrier handoff → customer delivery.

The timestamps are chronologically consistent, but their relationship with the canceled status remains unexplained. The available data does not establish when or why cancellation occurred. Source values have been preserved.

### **Order Items — Grain and Key Validation**

In [152]:
df_items = pd.read_csv(RAW_DATA_DIR/"olist_order_items_dataset.csv", dtype=str)
df_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [153]:
df_items.shape

(112650, 7)

In [154]:
df_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   order_id             112650 non-null  str  
 1   order_item_id        112650 non-null  str  
 2   product_id           112650 non-null  str  
 3   seller_id            112650 non-null  str  
 4   shipping_limit_date  112650 non-null  str  
 5   price                112650 non-null  str  
 6   freight_value        112650 non-null  str  
dtypes: str(7)
memory usage: 6.0 MB


In [155]:
df_items.shape

(112650, 7)

In [156]:
df_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

**Dataset Overview**

The order items dataset contains 112,650 rows and 7 columns.

The dataset grain is one order item per row. An order can contain multiple items.

**Composite Key Validation**

The combination of `order_id` and `order_item_id` uniquely identifies each row in the current dataset:

- No duplicate combinations were found.
- Neither key column contains missing values.

These checks support using (`order_id`, `order_item_id`) as the composite primary key candidate.


In [157]:
#Checking orphan records
df_items["order_id"].isin(df_orders["order_id"]).sum()
(~df_items["order_id"].isin(df_orders["order_id"])).sum()


np.int64(0)

## **Payments — Grain and Key Validation**

In [158]:
df_payments = pd.read_csv(RAW_DATA_DIR/"olist_order_payments_dataset.csv", dtype=str)
df_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [159]:
df_payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   order_id              103886 non-null  str  
 1   payment_sequential    103886 non-null  str  
 2   payment_type          103886 non-null  str  
 3   payment_installments  103886 non-null  str  
 4   payment_value         103886 non-null  str  
dtypes: str(5)
memory usage: 4.0 MB


In [160]:
df_payments.shape

(103886, 5)

In [161]:
df_payments["order_id"].value_counts().sort_values(ascending=False)
df_payments[df_payments["order_id"] == "fa65dad1b0e818e3ccc5cb0e39231352"].sort_values(by="payment_sequential")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86
32519,fa65dad1b0e818e3ccc5cb0e39231352,11,voucher,1,4.03
73537,fa65dad1b0e818e3ccc5cb0e39231352,12,voucher,1,3.60
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
63369,fa65dad1b0e818e3ccc5cb0e39231352,15,voucher,1,14.04
78050,fa65dad1b0e818e3ccc5cb0e39231352,16,voucher,1,12.86
79670,fa65dad1b0e818e3ccc5cb0e39231352,17,voucher,1,3.72
80321,fa65dad1b0e818e3ccc5cb0e39231352,18,voucher,1,0.73


In [162]:
df_payments.duplicated(subset=["order_id", "payment_sequential"]).sum()

np.int64(0)

In [163]:
(~df_payments["order_id"].isin(values=df_orders["order_id"])).sum()

np.int64(0)

In [164]:
df_payments["order_id"].isna().sum()
df_payments["payment_sequential"].isna().sum()


np.int64(0)

In [165]:
payment_values = pd.to_numeric(df_payments["payment_value"])
(payment_values < 0).sum()
(payment_values == 0).sum()
payment_values.isna().sum()

np.int64(0)

**Dataset Overview**

The payments dataset contains [row count] rows and [column count] columns.

The dataset grain is one payment record associated with an order per row. An order can have multiple payment records, including multiple records with the same payment method.

`payment_sequential` identifies each payment record within an order, while `payment_installments` describes the number of installments selected for that payment. A row does not represent an individual installment paid.

**Composite Key Validation**

The combination of `order_id` and `payment_sequential` contains no duplicate combinations.

Missing values in the candidate key columns:

- `order_id`: 0
- `payment_sequential`: 0

**Referential Integrity**

All `order_id` values in the payments dataset exist in the orders dataset. No orphan payment records were found.

**Payment Value Validation**

The `payment_value` column was checked for missing, negative, and zero values:

- Missing values: 0
- Negative values: 0
- Zero values: 9

Source values have been preserved. No payment records have been removed or modified during this analysis.